# RealVisXL V4.0 — Photorealistic Text-to-Image on Colab

Second attempt: same pipeline as `sdxl_colab.ipynb` but with **RealVisXL V4.0** — a fine-tuned SDXL checkpoint built for photorealism. Base SDXL produced painterly/3D-looking faces; RealVisXL should look like actual photographs.

**Runtime:** GPU (T4, free tier works)

**Model:** [SG161222/RealVisXL_V4.0](https://huggingface.co/SG161222/RealVisXL_V4.0)

## Cell 1 — Environment setup

In [ ]:
!pip install -q diffusers==0.30.0 transformers accelerate safetensors invisible_watermark mlflow ftfy
!pip install -q git+https://github.com/openai/CLIP.git

import os
os.makedirs('/content/outputs', exist_ok=True)

import torch
assert torch.cuda.is_available(), 'No GPU detected! Runtime → Change runtime type → T4 GPU'
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Cell 2 — Load RealVisXL V4.0

In [ ]:
from diffusers import StableDiffusionXLPipeline, DPMSolverMultistepScheduler
import torch

MODEL_ID = 'SG161222/RealVisXL_V4.0'

pipe = StableDiffusionXLPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    use_safetensors=True,
).to('cuda')

# DPM++ SDE Karras is the recommended sampler for RealVisXL
pipe.scheduler = DPMSolverMultistepScheduler.from_config(
    pipe.scheduler.config,
    use_karras_sigmas=True,
    algorithm_type='sde-dpmsolver++',
)

try:
    pipe.enable_xformers_memory_efficient_attention()
    print('xformers attention enabled')
except Exception as e:
    print(f'xformers not available ({e}), using SDPA fallback')

print(f'{MODEL_ID} loaded')

## Cell 3 — Load CLIP for metrics

In [ ]:
import clip

clip_model, clip_preprocess = clip.load('ViT-B/32', device='cuda')
clip_model.eval()
print('CLIP ViT-B/32 loaded')

## Cell 4 — Metrics (CLIP similarity, sharpness, diversity)

In [ ]:
import numpy as np
from PIL import Image
import torch.nn.functional as F
from scipy.ndimage import convolve


def clip_similarity(pil_image, prompt):
    image_input = clip_preprocess(pil_image).unsqueeze(0).to('cuda')
    text_tokens = clip.tokenize([prompt], truncate=True).to('cuda')
    with torch.no_grad():
        image_features = clip_model.encode_image(image_input)
        text_features = clip_model.encode_text(text_tokens)
        image_features /= image_features.norm(dim=-1, keepdim=True)
        text_features /= text_features.norm(dim=-1, keepdim=True)
    return (image_features * text_features).sum(dim=-1).item()


def sharpness(pil_image):
    arr = np.array(pil_image.convert('L'), dtype=np.float64)
    kernel = np.array([[0, 1, 0], [1, -4, 1], [0, 1, 0]], dtype=np.float64)
    return float(convolve(arr, kernel).var())


def pixel_diversity(pil_image):
    arr = np.array(pil_image, dtype=np.float32) / 255.0
    return float(arr.std())


print('Metrics ready: clip_similarity, sharpness, pixel_diversity')

## Cell 5 — MLflow setup

In [ ]:
import mlflow

mlflow.set_tracking_uri('sqlite:////content/mlflow.db')
mlflow.set_experiment('realvisxl_phase2')
print('MLflow tracking: /content/mlflow.db → experiment realvisxl_phase2')

## Cell 6 — Generation function

In [ ]:
import time


def generate(prompt, negative_prompt='', steps=30, guidance=7.5, seed=42,
             width=1024, height=1024):
    torch.cuda.reset_peak_memory_stats()
    gen = torch.Generator('cuda').manual_seed(seed)

    start = time.perf_counter()
    image = pipe(
        prompt,
        negative_prompt=negative_prompt,
        num_inference_steps=steps,
        guidance_scale=guidance,
        generator=gen,
        width=width,
        height=height,
    ).images[0]
    elapsed = time.perf_counter() - start
    vram_peak = torch.cuda.max_memory_allocated() / 1e9

    return image, elapsed, vram_peak


print('generate() ready')

## Cell 7 — Photorealistic portrait prompts + log to MLflow

RealVisXL responds better to photography-flavored prompts (lens, lighting, camera terms). The prompts are tuned accordingly.

Its negative-prompt recipe is also stronger — included below.

In [ ]:
PROMPTS = [
    'RAW photo, portrait of a young woman with red hair, natural skin texture, soft natural lighting, 85mm lens, shallow depth of field, ultra realistic',
    'RAW photo, portrait of an elderly man with white beard and long hair, wrinkles, skin pores, natural lighting, 85mm lens, ultra realistic',
    'RAW photo, portrait of a middle-aged man with blonde hair and blue eyes, natural skin texture, overcast lighting, 85mm lens, ultra realistic',
]

NEGATIVE = (
    '(worst quality, low quality, illustration, 3d, 2d, painting, cartoons, sketch), open mouth, '
    'deformed iris, deformed pupils, extra fingers, fewer fingers, bad anatomy, disfigured, '
    'bad proportions, duplicate, watermark, text, signature'
)

results = []

for prompt in PROMPTS:
    short = prompt.split(',')[1].strip() if ',' in prompt else prompt[:40]
    print(f'\nGenerating: {short}')

    with mlflow.start_run(run_name=short[:40]):
        image, elapsed, vram_peak = generate(
            prompt, negative_prompt=NEGATIVE, steps=30, guidance=4.0, seed=42,
        )

        clip_sim = clip_similarity(image, prompt)
        sharp = sharpness(image)
        diversity = pixel_diversity(image)

        filename = short.replace(' ', '_').replace(',', '')[:60] + '.png'
        path = f'/content/outputs/realvis_{filename}'
        image.save(path)

        mlflow.log_params({
            'prompt': prompt,
            'negative_prompt': NEGATIVE,
            'steps': 30,
            'guidance': 4.0,
            'seed': 42,
            'model': 'RealVisXL-V4.0',
            'scheduler': 'DPM++ SDE Karras',
        })
        mlflow.log_metrics({
            'clip_similarity': clip_sim,
            'sharpness': sharp,
            'pixel_diversity': diversity,
            'generation_time_s': elapsed,
            'vram_peak_gb': vram_peak,
        })
        mlflow.log_artifact(path)

        print(f'  CLIP sim: {clip_sim:.4f}  sharp: {sharp:.1f}  div: {diversity:.3f}  time: {elapsed:.1f}s  vram: {vram_peak:.1f}GB')
        results.append((short, image, clip_sim, sharp, diversity, elapsed))

print(f'\nGenerated {len(results)} images → /content/outputs/')

## Cell 8 — Display generated images

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(results), figsize=(6 * len(results), 6))
if len(results) == 1:
    axes = [axes]
for ax, (label, image, clip_sim, sharp, diversity, elapsed) in zip(axes, results):
    ax.imshow(image)
    ax.axis('off')
    ax.set_title(f'{label}\nCLIP: {clip_sim:.3f}  sharp: {sharp:.0f}  time: {elapsed:.1f}s', fontsize=10)
plt.tight_layout()
plt.show()

## Cell 9 — Parameter sweep on RealVisXL

RealVisXL typically works best at **guidance 3–5** (lower than SDXL Base's 7.5) and **steps 25–40**.
This sweep finds the optimal combo for this model specifically — different from SDXL Base's optimal.

In [ ]:
SWEEP_PROMPT = 'RAW photo, portrait of a young woman with red hair, natural skin texture, soft natural lighting, 85mm lens, shallow depth of field, ultra realistic'

sweep_results = []

for guidance in [3.0, 4.0, 5.0, 7.0]:
    for steps in [25, 30, 40]:
        print(f'guidance={guidance}  steps={steps}')

        with mlflow.start_run(run_name=f'sweep_g{guidance}_s{steps}'):
            image, elapsed, vram_peak = generate(
                SWEEP_PROMPT, negative_prompt=NEGATIVE,
                steps=steps, guidance=guidance, seed=42,
            )
            clip_sim = clip_similarity(image, SWEEP_PROMPT)
            sharp = sharpness(image)
            diversity = pixel_diversity(image)

            filename = f'realvis_sweep_g{guidance}_s{steps}.png'
            path = f'/content/outputs/{filename}'
            image.save(path)

            mlflow.log_params({
                'prompt': SWEEP_PROMPT,
                'steps': steps,
                'guidance': guidance,
                'seed': 42,
                'sweep': True,
                'model': 'RealVisXL-V4.0',
            })
            mlflow.log_metrics({
                'clip_similarity': clip_sim,
                'sharpness': sharp,
                'pixel_diversity': diversity,
                'generation_time_s': elapsed,
            })
            mlflow.log_artifact(path)

            sweep_results.append((guidance, steps, image, clip_sim, sharp))

print(f'\nSweep complete: {len(sweep_results)} combos')

## Cell 10 — Sweep results grid

In [ ]:
import matplotlib.pyplot as plt

guidance_values = sorted(set(g for g, _, _, _, _ in sweep_results))
steps_values = sorted(set(s for _, s, _, _, _ in sweep_results))

fig, axes = plt.subplots(len(guidance_values), len(steps_values),
                         figsize=(4 * len(steps_values), 4 * len(guidance_values)))

lookup = {(g, s): (img, clip_sim, sharp) for g, s, img, clip_sim, sharp in sweep_results}

for i, g in enumerate(guidance_values):
    for j, s in enumerate(steps_values):
        ax = axes[i, j] if len(guidance_values) > 1 else axes[j]
        img, clip_sim, sharp = lookup[(g, s)]
        ax.imshow(img)
        ax.axis('off')
        ax.set_title(f'g={g}  s={s}\nCLIP: {clip_sim:.3f}  sharp: {sharp:.0f}', fontsize=9)

plt.tight_layout()
plt.show()

## Cell 11 — Launch MLflow UI (optional)

Both experiments (`sdxl_phase1` and `realvisxl_phase2`) share the same SQLite DB, so you can compare side by side in one UI.

In [ ]:
!pip install -q pyngrok
!pkill -f 'mlflow ui' || true

import subprocess, time
mlflow_proc = subprocess.Popen(
    ['mlflow', 'ui', '--backend-store-uri', 'sqlite:////content/mlflow.db', '--port', '5000'],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
time.sleep(3)

from pyngrok import ngrok
# ngrok.set_auth_token('YOUR_TOKEN_HERE')  # optional, for persistent tunnels
public_url = ngrok.connect(5000)
print(f'MLflow UI: {public_url}')